### Práctica 2: Localización de un robot con diferentes sensores

Generación de trayectorias sintéticas para el robot.

Crea archivos CSV con los siguientes datos:
- t: Tiempo
- x, y: Posición
- theta: Orientación
- vx, vy: Velocidad
- omega: Velocidad angular

In [2]:
import numpy as np
import pandas as pd
import json
from pathlib import Path

Trayectoria circular:

- t_max: Tiempo total
- dt: Paso temporal
- r: Radio de la circunferencia
- omega: Velocidad angular

In [3]:
def generate_circular_path(t_max=60.0, dt=0.05, r=10.0, omega=0.3):
    t = np.arange(0, t_max, dt)
    theta = omega * t
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    vx = -r * omega * np.sin(theta)
    vy = r * omega * np.cos(theta)
    return pd.DataFrame({
        't': t, 'x': x, 'y': y, 'theta': theta,
        'vx': vx, 'vy': vy, 'omega': np.full_like(t, omega)
    })

Trayectoria cuadrada:

- t_max: Tiempo total
- dt: Paso temporal
- l: Lado del cuadrado

In [4]:
def generate_square_path(t_max=60.0, dt=0.05, l=10.0):
    t = np.arange(0, t_max, dt)
    n = len(t)
    v = 4 * l / t_max
    vertices = np.array([[0.0, 0.0],[l, 0.0],[l, l],[0.0, l],[0.0, 0.0]])  
    lengths = np.array([0.0, l, 2*l, 3*l, 4*l])
    s = v * t
    x = np.zeros(n)
    y = np.zeros(n)
    for i, si in enumerate(s):
        si = min(si, 4*l) 
        idx = np.searchsorted(lengths, si, side='right') - 1
        idx = min(idx, 3)  # máximo lado 3
        s_local = si - lengths[idx]
        p0 = vertices[idx]
        p1 = vertices[idx + 1]
        dir = p1 - p0
        norm = np.linalg.norm(dir)
        if norm > 0:
            u = dir / norm
        else:
            u = np.array([0.0, 0.0])
        x[i] = p0[0] + u[0] * s_local
        y[i] = p0[1] + u[1] * s_local
    
    theta = np.zeros(n)
    for i, si in enumerate(s):
        si = min(si, 4*l - 1e-6)
        idx = np.searchsorted(lengths, si, side='right') - 1
        idx = min(idx, 3)
        p0 = vertices[idx]
        p1 = vertices[idx + 1]
        direccion = p1 - p0
        theta[i] = np.arctan2(direccion[1], direccion[0])
    
    vx = v * np.cos(theta)
    vy = v * np.sin(theta)
    omega = np.gradient(theta, dt)
    ventana = 5
    kernel = np.ones(ventana) / ventana
    omega = np.convolve(omega, kernel, mode='same')
    
    return pd.DataFrame({
        't': t, 'x': x, 'y': y, 'theta': theta,
        'vx': vx, 'vy': vy, 'omega': omega
    })

Trayectoria en ocho:

- t_max: Tiempo total
- dt: Paso temporal
- scale: Escala

In [7]:
def generate_eight_path(t_max=60.0, dt=0.05, scale=10.0):
    """Trayectoria en forma de 8: más exigente para el filtro."""
    t = np.arange(0, t_max, dt)
    x = scale * np.sin(0.3 * t)
    y = scale * np.sin(0.6 * t)
    vx = scale * 0.3 * np.cos(0.3 * t)
    vy = scale * 0.6 * np.cos(0.6 * t)
    theta = np.arctan2(vy, vx)
    omega = np.gradient(theta, dt)
    return pd.DataFrame({
        't': t, 'x': x, 'y': y, 'theta': theta,
        'vx': vx, 'vy': vy, 'omega': omega
    })

Trayectoria urbana:

- t_max: Tiempo total
- dt: Paso temporal

In [8]:
def generate_urban_path(t_max=60.0, dt=0.05):
    """Trayectoria tipo urbana: rectas, giros y curvas."""
    t = np.arange(0, t_max, dt)
    x = np.zeros_like(t)
    y = np.zeros_like(t)
    theta = np.zeros_like(t)
    v = 2.0  # velocidad lineal m/s
    for i in range(1, len(t)):
        if i < 200:
            omega = 0.0
        elif i < 400:
            omega = 0.5
        elif i < 600:
            omega = 0.0
        elif i < 800:
            omega = -0.4
        else:
            omega = 0.0
        theta[i] = theta[i-1] + omega * dt
        x[i] = x[i-1] + v * np.cos(theta[i]) * dt
        y[i] = y[i-1] + v * np.sin(theta[i]) * dt
    vx = v * np.cos(theta)
    vy = v * np.sin(theta)
    omega_arr = np.gradient(theta, dt)
    return pd.DataFrame({
        't': t, 'x': x, 'y': y, 'theta': theta,
        'vx': vx, 'vy': vy, 'omega': omega_arr
    })

Generación de archivos:

1. Trayectoria circular
2. Trayectoria cuadrada
3. Trayectoria en ocho
4. Trayectoria urbana

**TODO: Puedes crear tus propias trayectorias para el análisis de resultados**

In [11]:
Path("data").mkdir(exist_ok=True)

trayectorias = {
    'trayectoria_circular': generate_circular_path(),
    'trayectoria_cuadrada': generate_square_path(),
    'trayectoria_ocho': generate_eight_path(),
    'trayectoria_urbana': generate_urban_path(),
}

for nombre, df in trayectorias.items():
    df.to_csv(f"data/{nombre}_gt.csv", index=False)
    meta = {
        'nombre': nombre,
        'frecuencia_hz': 1.0 / (df['t'].iloc[1] - df['t'].iloc[0]),
        'unidades': {'x': 'm', 'y': 'm', 'theta': 'rad', 'vx': 'm/s', 'vy': 'm/s', 'omega': 'rad/s'},
        'fuente': 'sintetica',
        'robot': 'diferencial'
    }
    with open(f"data/{nombre}_meta.json", 'w') as f:
        json.dump(meta, f, indent=2)